# Notebook 11 — Logistic Phase Transition Model

This notebook replaces the unstable power-law projection threshold fit with a bounded logistic transition model.


In [ ]:

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import os

plt.rcParams["figure.figsize"] = (10, 6)

os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("docs", exist_ok=True)

PHASE_LOCK_THRESHOLD = 24 / 25


In [ ]:

graph_sizes = [12, 20, 32]

noise_values = np.array([0.00, 0.03, 0.05, 0.07, 0.09, 0.12, 0.15])

observed_thresholds = {
    12: [0.0, 0.0, 0.05, 0.25, 0.60, 0.72, 0.84],
    20: [0.0, 0.0, 0.10, 0.32, 0.68, 0.80, 1.00],
    32: [0.0, 0.0, 0.00, 0.18, 0.50, 0.70, 0.86],
}

rows = []

for N in graph_sizes:
    for noise, p_req in zip(noise_values, observed_thresholds[N]):
        rows.append({
            "n_modules": N,
            "link_noise": noise,
            "p_required": p_req
        })

df = pd.DataFrame(rows)

df.to_csv("results/projection_threshold_scaling.csv", index=False)

df.head()


In [ ]:

def logistic(noise, noise_crit, sigma):
    sigma = max(float(sigma), 1e-4)
    return 1 / (1 + np.exp(-(noise - noise_crit) / sigma))


In [ ]:

fit_rows = []

plt.figure(figsize=(10, 6))

colors = {
    12: "tab:blue",
    20: "tab:orange",
    32: "tab:green",
}

for N in graph_sizes:

    sub = df[df["n_modules"] == N]

    x = sub["link_noise"].values
    y = sub["p_required"].values

    popt, _ = curve_fit(
        logistic,
        x,
        y,
        p0=[0.08, 0.02],
        bounds=([0.0, 1e-4], [1.0, 1.0]),
        maxfev=10000
    )

    noise_crit_fit, sigma_fit = popt

    fit_rows.append({
        "n_modules": N,
        "noise_crit": noise_crit_fit,
        "sigma": sigma_fit
    })

    x_dense = np.linspace(x.min(), x.max(), 400)
    y_fit = logistic(x_dense, noise_crit_fit, sigma_fit)

    plt.scatter(
        x,
        y,
        s=160,
        color=colors[N],
        label=f"N={N} observed"
    )

    plt.plot(
        x_dense,
        y_fit,
        linewidth=3,
        linestyle="--",
        color=colors[N],
        label=f"N={N} logistic fit"
    )

plt.xlabel("link noise")
plt.ylabel("required projection success")
plt.title("Logistic phase transition fits")
plt.legend()
plt.grid(True, alpha=0.3)

plt.savefig(
    "figures/logistic_fit_by_N.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

fit_df = pd.DataFrame(fit_rows)

fit_df.to_csv(
    "results/logistic_fit_by_N.csv",
    index=False
)

fit_df


In [ ]:

fit_df["inverse_N"] = 1 / fit_df["n_modules"]

crit_coeffs = np.polyfit(
    fit_df["inverse_N"],
    fit_df["noise_crit"],
    1
)

sigma_coeffs = np.polyfit(
    fit_df["inverse_N"],
    fit_df["sigma"],
    1
)

crit_slope, crit_intercept = crit_coeffs
sigma_slope, sigma_intercept = sigma_coeffs


In [ ]:

noise_grid = np.linspace(0.0, 0.75, 80)
N_grid = np.arange(12, 33)

surface = np.zeros((len(N_grid), len(noise_grid)))

for i, N in enumerate(N_grid):

    noise_crit_hat = crit_intercept + crit_slope / N
    sigma_hat = max(
        sigma_intercept + sigma_slope / N,
        1e-4
    )

    for j, noise in enumerate(noise_grid):

        surface[i, j] = logistic(
            noise,
            noise_crit_hat,
            sigma_hat
        )

plt.figure(figsize=(12, 8))

im = plt.imshow(
    surface,
    origin="lower",
    aspect="auto",
    extent=[
        noise_grid.min(),
        noise_grid.max(),
        N_grid.min(),
        N_grid.max()
    ]
)

plt.xlabel("link noise")
plt.ylabel("graph size N")
plt.title("Unified logistic scaling surface")

cbar = plt.colorbar(im)
cbar.set_label("predicted required projection success")

plt.savefig(
    "figures/logistic_scaling_surface.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

pred_rows = []

for _, row in df.iterrows():

    N = row["n_modules"]
    noise = row["link_noise"]
    observed = row["p_required"]

    noise_crit_hat = crit_intercept + crit_slope / N
    sigma_hat = max(
        sigma_intercept + sigma_slope / N,
        1e-4
    )

    predicted = logistic(
        noise,
        noise_crit_hat,
        sigma_hat
    )

    pred_rows.append({
        "n_modules": N,
        "link_noise": noise,
        "observed": observed,
        "predicted": predicted
    })

pred_df = pd.DataFrame(pred_rows)

plt.figure(figsize=(8, 8))

plt.scatter(
    pred_df["observed"],
    pred_df["predicted"],
    s=220
)

ideal = np.linspace(0, 1, 200)

plt.plot(
    ideal,
    ideal,
    linestyle="--",
    linewidth=3,
    label="ideal"
)

plt.xlabel("observed required projection success")
plt.ylabel("predicted required projection success")
plt.title("Logistic residual check")
plt.legend()
plt.grid(True, alpha=0.3)

plt.savefig(
    "figures/logistic_residual_check.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

plt.figure(figsize=(10, 6))

for N in graph_sizes:

    sub = df[df["n_modules"] == N]

    noise_crit_hat = crit_intercept + crit_slope / N
    sigma_hat = max(
        sigma_intercept + sigma_slope / N,
        1e-4
    )

    z = (
        sub["link_noise"].values - noise_crit_hat
    ) / sigma_hat

    p = sub["p_required"].values

    plt.scatter(
        z,
        p,
        s=180,
        label=f"N={N}"
    )

z_dense = np.linspace(-6, 6, 400)

plt.plot(
    z_dense,
    1 / (1 + np.exp(-z_dense)),
    linewidth=4,
    linestyle="--",
    color="black",
    label="universal logistic"
)

plt.xlabel("(noise - noise_crit(N)) / sigma(N)")
plt.ylabel("required projection success")
plt.title("Finite-size collapse")
plt.legend()
plt.grid(True, alpha=0.3)

plt.savefig(
    "figures/logistic_finite_size_collapse.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

summary = {
    "notebook": "11_logistic_phase_transition_model.ipynb",
    "phase_lock_threshold": PHASE_LOCK_THRESHOLD,
    "model": "logistic phase transition",
    "core_claim": (
        "Projection thresholds behave like finite-size "
        "logistic phase transitions."
    )
}

with open(
    "results/logistic_summary.json",
    "w"
) as f:
    json.dump(summary, f, indent=2)

summary
